# 03c — Markov-Switching Dynamic Regression

Dette notebooket trener en 2-regime MS-DR-modell på samme ISO-grunnlag.
For å holde modellen lett og stabil brukes bare basisvariablene som eksogene forklaringsvariabler.

**Input:** `intermediate/df_iso.parquet`

**Output:** `intermediate/preds_msdr.parquet`, `intermediate/models_msdr.pkl`


In [1]:
import pandas as pd
from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression

from src.config import (
    INTERMEDIATE_DIR, TARGET, MSDR_FEATURES, TRAIN_YEARS, TEST_YEARS, apply_style
)
from src.evaluation import eval_metrics
from src.model_training import (
    load_prepared_iso_data, split_features_target, make_prediction_frame,
    save_model_artifacts, predict_msdr_out_of_sample,
)

apply_style()


In [2]:
df_iso = load_prepared_iso_data(INTERMEDIATE_DIR)
X_train, y_train, X_test, y_test, train_mask, test_mask = split_features_target(
    df_iso,
    feature_cols=MSDR_FEATURES,
    target=TARGET,
    train_years=TRAIN_YEARS,
    test_years=TEST_YEARS,
)

print(f"Trening: {len(X_train):,} ISO-timer")
print(f"Test:    {len(X_test):,} ISO-timer")
print(f"Features: {MSDR_FEATURES}")


Trening: 25,072 ISO-timer
Test:    9,844 ISO-timer
Features: ['cons_NO4', 'fill_avvik', 'prod_wind_NO4']


In [3]:
msdr = MarkovRegression(
    y_train,
    k_regimes=2,
    trend="c",
    exog=X_train,
    switching_variance=True,
)
msdr_result = msdr.fit(maxiter=200, disp=False)

print(msdr_result.summary())


/Library/Frameworks/Python.framework/Versions/3.8/lib/python3.8/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


                        Markov Switching Model Results                        
Dep. Variable:              price_NO4   No. Observations:                25072
Model:               MarkovRegression   Log Likelihood             -156898.860
Date:                Fri, 29 May 2026   AIC                         313821.720
Time:                        14:39:52   BIC                         313919.274
Sample:                             0   HQIC                        313853.290
                              - 25072                                         
Covariance Type:               approx                                         
                             Regime 0 parameters                              
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const       -192.5861      3.551    -54.229      0.000    -199.547    -185.626
x1             0.1470      0.002     86.054      0.0

In [4]:
msdr_pred, ergodic_probs, transition_matrix = predict_msdr_out_of_sample(
    msdr_result,
    y_train=y_train,
    y_test=y_test,
    X_train=X_train,
    X_test=X_test,
    probabilities="predicted",
)

preds = make_prediction_frame(
    df=df_iso,
    mask=test_mask,
    actual=y_test,
    prediction_col="MSDR",
    predictions=msdr_pred,
)
metrics = eval_metrics(preds["actual"], preds["MSDR"])

display(pd.DataFrame([metrics], index=["MSDR"]))
print("Ergodiske regimesannsynligheter:")
print(pd.Series(ergodic_probs, index=["Regime 0", "Regime 1"]).round(4))


/Library/Frameworks/Python.framework/Versions/3.8/lib/python3.8/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


,MAE,RMSE,R²,N
MSDR,118.3,183.6,0.4856,9844


Ergodiske regimesannsynligheter:
Regime 0    0.5
Regime 1    0.5
dtype: float64


In [5]:
payload = {
    "model_name": "MSDR",
    "prediction_col": "MSDR",
    "feature_cols": MSDR_FEATURES,
    "target_col": TARGET,
    "train_years": TRAIN_YEARS,
    "test_years": TEST_YEARS,
    "estimator": msdr_result,
    "metrics": metrics,
    "ergodic_probs": ergodic_probs,
    "transition_matrix": transition_matrix,
}

save_model_artifacts("msdr", preds, payload, intermediate_dir=INTERMEDIATE_DIR)
print(f"Lagret {INTERMEDIATE_DIR}preds_msdr.parquet")
print(f"Lagret {INTERMEDIATE_DIR}models_msdr.pkl")


Lagret intermediate/preds_msdr.parquet
Lagret intermediate/models_msdr.pkl
